---
title: Agent-based Music Recommendation
authors:
  - name: SeungHeon Doh
---

The [search and recommendation page](./search-recommendation.md) ended with the LLM as an agent: the catalog and the listening data stay in tools, and the LLM decides which tools to call, in what order, and with what arguments. This notebook builds that agent, following TalkPlay-Tools {cite}`doh_talkplaytools_2025`. The LLM reads a request, plans a retrieval call and a reranking call, the tools run one after the other on a shrinking set of candidates, and the LLM writes the reply around the track that comes out on top. The catalog is the test split of TalkPlayData-2 {cite}`doh_talkplaydata2_2025`, with every index and embedding the tools need computed in advance.

The notebook builds the system in three layers and runs it once at the end. The **environment** is the data and the tools that read it. The **workflow** is how tool calls are chained. The **agent** is the prompts, the two LLM calls of each turn, and the memory between turns.

## What we will build

1. **Design the environment.** Look at the catalog and the indices computed from it in advance, then write six tools as Python functions over them. Their docstrings are what the LLM reads.
2. **Design the workflow.** Chain tool calls by hand, a retrieval call and then a reranking call, each one searching only what the previous one returned.
3. **Design the agent.** Assemble the prompts from a role, a listener profile and a goal, and wire two LLM calls of Qwen3 {cite}`yang_qwen3_2025` around the executor.
4. **Run it** on one request.

:::{figure} img/search-talkplay-tools.png
:label: fig-demo-talkplay-tools
:alt: The TalkPlay-Tools agent. A user query and profile go to an LLM that plans tool calls. An executor runs the tools (SQL, BM25, text-to-item, item-to-item, user-to-item, and semantic ID matching) against the music and user databases. The LLM reranks the retrieved tracks and writes the response.

The system this notebook builds: the LLM plans tool calls, an executor runs them over the music and user databases, and the LLM writes the reply from the results. Figure from {cite}`doh_talkplaytools_2025`.
:::

## Prerequisites

Run this notebook on a Colab GPU runtime; a T4 is enough. The tools take milliseconds, because everything they search was computed in advance. The LLM is `Qwen/Qwen3-4B`, the model of the TalkPlay-Tools repository, and most of each request is spent thinking before the tool calls.

The tool environment is a 2.4 GB download from the Hugging Face Hub. Most of it is the CLAP audio encoder, which this notebook does not load, so it is skipped when the archive is unpacked. The catalog and the environment are released for non-commercial research use (CC BY-NC 4.0), and the tracks come with Spotify IDs, not audio.

## Step 1: Setting up

In [ ]:
%pip install -q bm25s sqlparse transformers accelerate pandas

In [1]:
import json, os, random, re, sqlite3, tarfile, textwrap, time, warnings

warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")

import bm25s
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from transformers.utils import get_json_schema, logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
LM_ID = "Qwen/Qwen3-4B"
LM_DTYPE = {"cuda": torch.bfloat16, "mps": torch.float16, "cpu": torch.float32}[DEVICE]
print("device:", DEVICE, "| LLM:", LM_ID)

device: mps | LLM: Qwen/Qwen3-4B


The archive unpacks into a `cache` folder with one subfolder per tool.

In [2]:
ENV = "cache"
if not os.path.exists(f"{ENV}/sql/tracks.db"):
    archive = hf_hub_download("talkpl-ai/TalkPlayTools-Env", "test_env.tar.gz", repo_type="dataset")
    with tarfile.open(archive) as tar:
        members = [m for m in tar.getmembers() if "music_audioset" not in m.name]  # skip the CLAP checkpoint
        tar.extractall(".", members=members, filter="data")

for folder in sorted(os.listdir(ENV)):
    print(f"{ENV}/{folder}:", ", ".join(sorted(os.listdir(f"{ENV}/{folder}"))))

cache/bm25: album, artist, attributes, lyrics, title, track_index.json
cache/encoder: models, vector_db.pt
cache/metadata: item_metadata.json, user_profiles.json
cache/semantic_id: rvq_r4_c64
cache/sql: tracks.db


## Step 2: Designing the environment

The environment is everything the agent can reach. It has two parts: data, prepared once, and tools, which the LLM calls. All the expensive work, embedding audio, images and text, training the collaborative-filtering model, quantizing embeddings, happens when the data is prepared, so that a tool call at query time is a lookup, a filter or a dot product.

:::{figure} ../assets/tools-environment.svg
:label: fig-tools-environment
:alt: The catalog of 6,761 tracks and the listening logs of 18,090 listeners are turned in advance into a SQL table, BM25 indices over five text fields, embeddings of six views, semantic IDs and listener vectors. Six tools read them: sql, bm25, text_to_item_similarity, item_to_item_similarity, semantic_id_matching and user_to_item_similarity.

The environment. Data is indexed once; each tool reads one kind of index and searches only the current pool of candidates.
:::

Each track has a record of metadata, tags, a caption written by a model, tempo, key, and lyrics. The function below turns a record into the one-line description that the LLM will read, as in the repository.

In [3]:
catalog = json.load(open(f"{ENV}/metadata/item_metadata.json"))
users = json.load(open(f"{ENV}/metadata/user_profiles.json"))

def describe(track_id, semantic_ids=False):
    t = catalog[track_id]
    s = (f"track_id: {track_id}, title: {t['track_name'][0].lower()}, "
         f"artist: {', '.join(t['artist_name']).lower()}, album: {', '.join(t['album_name']).lower()}")
    if t["tag_list"]:
        s += f", tags: {', '.join(t['tag_list'])}"
    if t["tempo"]:
        s += f", tempo: {float(t['tempo'][0])}"
    if t["key"]:
        s += f", key: {t['key'][0]}"
    if t["release_date"]:
        s += f", release_date: {t['release_date']}"
    if semantic_ids:
        s += ", " + ", ".join(f"{m}:semantic_id: {SEMANTIC_IDS[m][track_id]}" for m in SEMANTIC_IDS if track_id in SEMANTIC_IDS[m])
    return s

print(len(catalog), "tracks,", len(users), "users")
pd.DataFrame([
    {"title": t["track_name"][0], "artist": ", ".join(t["artist_name"]), "tags": ", ".join(t["tag_list"][:6]),
     "tempo": t["tempo"][0] if t["tempo"] else None, "key": t["key"][0] if t["key"] else None}
    for t in list(catalog.values())[:5]
])

6761 tracks, 500 users


,title,artist,tags,tempo,key
0,Circle,Jon Hopkins,"easy listening, somafm, lounge, ambient, downtempo, instrumental",86.04,E major
1,Ventolin (Video Edit),Aphex Twin,"experimental, electronic, ambient, electronic, jazz, alternative",72.87,Db major
2,Dig Up Her Bones,Misfits,"punk, loved, alternative rock, favorites, alternative, good stuff",110.96,Db major
3,Everything You Do Is A Balloon,Boards of Canada,"experimental, favorites, heard on pandora, indie, alternative, summer",91.43,C major
4,503,"Hans Zimmer, Nick Glennie-Smith, Joshua Bell","beautiful, fucking awesome, amazing, epic, dreamy, melancholy",61.69,F major


Three kinds of index were built from these records ahead of time. **Embeddings** place every track in a vector space per view: the metadata line, the lyrics and the tags embedded with Qwen3-Embedding, the audio with CLAP, the cover art with SigLIP2, and the listening data with a collaborative-filtering model, which also gives every listener a vector. **Semantic IDs** compress each embedding into four integer codes with residual vector quantization, so that a track can be named in text by its codes. **A SQL table and BM25 indices** hold the facts and the words.

In [4]:
vector_db = torch.load(f"{ENV}/encoder/vector_db.pt", map_location="cpu")
USER_VECTORS = torch.load(f"{ENV}/encoder/models/cf_model.pt", map_location="cpu")["model"]["user"]  # item vectors are in vector_db
SEMANTIC_IDS = {m: torch.load(f"{ENV}/semantic_id/rvq_r4_c64/indices/{m}.pt") for m in
                ["audio", "image", "metadata", "lyrics", "attributes", "cf_item"]}

# one matrix per view, rows in a fixed track order
VECTORS = {}
for view, table in vector_db.items():
    ids = list(table)
    VECTORS[view] = (ids, torch.stack([table[i].float().view(-1) for i in ids]))
del vector_db

first = next(iter(catalog))
pd.DataFrame([
    {"view": view, "tracks": len(ids), "dimensions": mat.shape[1]} for view, (ids, mat) in VECTORS.items()
] + [{"view": "listeners (collaborative filtering)", "tracks": len(USER_VECTORS), "dimensions": 128}])

,view,tracks,dimensions
0,metadata,6761,1024
1,lyrics,6761,1024
2,attributes,6761,1024
3,audio,6761,512
4,image,6761,768
5,cf,6637,128
6,listeners (collaborative filtering),18090,128


In [5]:
print(describe(first, semantic_ids=True))

track_id: 0nfhGk3Cwp1qIu0C5sgI4S, title: circle, artist: jon hopkins, album: contact note, tags: easy listening, somafm, lounge, ambient, downtempo, instrumental, dreamy, chillout, melodic, chill, good stuff, good, electronica, electronic, tempo: 86.04, key: E major, release_date: 2005-05-16, audio:semantic_id: [0, 46, 5, 44], image:semantic_id: [41, 30, 32, 17], metadata:semantic_id: [38, 48, 7, 2], lyrics:semantic_id: [56, 6, 12, 33], attributes:semantic_id: [20, 4, 6, 33], cf_item:semantic_id: [51, 34, 33, 16]


## Step 3: Tools, the interface of the environment

A tool is a Python function. The LLM never sees its code, only a JSON schema generated from its signature and docstring: the name, what it is good for, and its arguments. Writing a tool well therefore means writing the docstring well, and the docstrings below follow those of the repository, including its advice on which stage each tool suits.

The tools share one rule. A search runs only over the current **pool** of candidates, so that a second tool reranks what the first one found instead of searching the whole catalog again.

In [6]:
POOL = None  # the track_ids the next tool may return; None means the whole catalog

def in_pool(track_id):
    return POOL is None or track_id in POOL

DB = sqlite3.connect(f"{ENV}/sql/tracks.db")
BM25 = {c: bm25s.BM25.load(f"{ENV}/bm25/{c}") for c in ["title", "artist", "album", "lyrics", "attributes"]}
BM25_IDS = json.load(open(f"{ENV}/bm25/track_index.json"))

embed_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")
embed_model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B", dtype=torch.bfloat16).eval()

@torch.no_grad()
def embed_text(text):
    x = embed_tokenizer([text], return_tensors="pt", truncation=True)
    return embed_model(**x).last_hidden_state[0, -1].float()  # last-token pooling, as the stored vectors were made

def nearest(view, query_vector, topk):
    ids, mat = VECTORS[view]
    order = torch.argsort(mat @ query_vector, descending=True).tolist()
    return [ids[i] for i in order if in_pool(ids[i])][:topk]

In [7]:
def sql(sql_query: str, topk: int) -> list:
    """
    Run an SQL query for boolean filtering on facts. Good for the retrieval stage. The query must return track_id (SELECT track_id FROM tracks WHERE ...).
    Schema: tracks(track_id TEXT, title TEXT, artist TEXT, album TEXT, popularity INTEGER from 0 to 93, release_date TEXT 'YYYY-MM-DD' from 1900 to 2017, tempo REAL in BPM from 41 to 208, key TEXT such as 'G major' or 'D# minor').

    Args:
        sql_query: The SQL query.
        topk: The number of track_ids to return.
    """
    rows = DB.execute(sql_query.strip().rstrip(";")).fetchall()
    return [r[0] for r in rows if in_pool(r[0])][:topk]

def bm25(query: str, corpus_type: str, topk: int) -> list:
    """
    BM25 search for lexical matching, in lowercase. Good for the retrieval stage, and for names and words that must appear.
    The corpora: title, artist and album names; lyrics; attributes (genre, instrument, mood, theme and usage tags).

    Args:
        query: The search query.
        corpus_type: One of "title", "artist", "album", "lyrics", "attributes".
        topk: The number of track_ids to return.
    """
    tokens = bm25s.tokenize(query.lower(), return_ids=False, show_progress=False)[0]
    scores = BM25[corpus_type].get_scores(tokens)
    order = scores.argsort()[::-1]
    return [BM25_IDS[i] for i in order if scores[i] > 0 and in_pool(BM25_IDS[i])][:topk]

def text_to_item_similarity(query: str, corpus_type: str, topk: int) -> list:
    """
    Dense retrieval: embed the query and find tracks whose metadata, lyrics or attributes are closest in meaning.
    Good for the retrieval and the reranking stage, and for descriptive queries (moods, situations, styles).

    Args:
        query: The search query.
        corpus_type: One of "metadata", "lyrics", "attributes".
        topk: The number of track_ids to return.
    """
    return nearest(corpus_type, embed_text(query), topk)

def item_to_item_similarity(track_id: str, corpus_type: str, topk: int) -> list:
    """
    Find tracks similar to a given track, by how it sounds (audio), how its cover looks (image), or who listens to it (cf).
    Good for the retrieval and the reranking stage, and for requests like "more like this". track_id is a 22-character string.

    Args:
        track_id: The seed track.
        corpus_type: One of "audio", "image", "cf".
        topk: The number of track_ids to return.
    """
    ids, mat = VECTORS[corpus_type]
    return nearest(corpus_type, mat[ids.index(track_id)], topk)

def user_to_item_similarity(user_id: str, topk: int) -> list:
    """
    Rank tracks by a collaborative-filtering model of the listener's history. Good for personalization, in the reranking stage.
    Use only the user_id from the demographic information. If user_type is "cold_start", do not select this tool.

    Args:
        user_id: The listener.
        topk: The number of track_ids to return.
    """
    return nearest("cf", USER_VECTORS[user_id].float().view(-1), topk)

def semantic_id_matching(modality_type: str, indices: list[int], topk: int) -> list:
    """
    Find tracks whose semantic ID, four integer codes, matches the given codes in the most positions.
    Good for continuing from tracks listed with their semantic IDs, such as a listener's previous session.

    Args:
        modality_type: One of "audio", "image", "metadata", "lyrics", "attributes", "cf_item".
        indices: The four codes.
        topk: The number of track_ids to return.
    """
    codes = SEMANTIC_IDS[modality_type]
    score = {t: sum(a == b for a, b in zip(c, indices)) for t, c in codes.items() if in_pool(t)}
    return sorted(score, key=score.get, reverse=True)[:topk]

TOOLS = {f.__name__: f for f in [sql, bm25, text_to_item_similarity, item_to_item_similarity,
                                 user_to_item_similarity, semantic_id_matching]}
SCHEMAS = [get_json_schema(f) for f in TOOLS.values()]
print(json.dumps(SCHEMAS[1], indent=2))

{
  "type": "function",
  "function": {
    "name": "bm25",
    "description": "BM25 search for lexical matching, in lowercase. Good for the retrieval stage, and for names and words that must appear.\nThe corpora: title, artist and album names; lyrics; attributes (genre, instrument, mood, theme and usage tags).",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "The search query."
        },
        "corpus_type": {
          "type": "string",
          "description": "One of \"title\", \"artist\", \"album\", \"lyrics\", \"attributes\"."
        },
        "topk": {
          "type": "integer",
          "description": "The number of track_ids to return."
        }
      },
      "required": [
        "query",
        "corpus_type",
        "topk"
      ]
    },
    "return": {
      "type": "array"
    }
  }
}


## Step 4: Designing the workflow

The workflow decides how tool calls combine. TalkPlay-Tools uses a cascade of two stages. A **retrieval** call casts a wide net over the whole catalog, and a **reranking** call orders what it caught. Each call searches only the pool the previous one returned, so the two tools should be complementary: a filter for the facts in a request and a similarity search for its description, for example.

:::{figure} ../assets/tools-workflow.svg
:label: fig-tools-workflow
:alt: The catalog of 6,761 tracks enters a retrieval call, sql with tempo under 80, which keeps 1,000 tracks. They become the pool of a reranking call, text_to_item_similarity with calm piano over the tags, which returns 20 ranked tracks; the first goes to the reply. Reversed, the dense call keeps 20 and the filter returns 13 of them in table order.

The workflow. Retrieval, then reranking, over a shrinking pool. A failed or empty call is skipped and the pool kept.
:::

Before handing the tools to the LLM, we chain them ourselves. A tool call is a name and a dictionary of arguments, exactly what the LLM will write. The executor runs the calls in order, and each successful call becomes the pool of the next one. A call that fails or returns nothing is recorded and skipped, so one bad call does not empty the result.

In [8]:
def run_tools(calls):
    global POOL
    POOL, ranking, steps = None, [], []
    for call in calls:
        try:
            result, error = TOOLS[call["name"]](**call["arguments"]), ""
        except Exception as e:
            result, error = [], f"{type(e).__name__}: {e}"
        if result:
            ranking, POOL = result, set(result)
        steps.append({"tool": call["name"], "arguments": call["arguments"], "returned": len(result), "error": error})
    POOL = None
    return ranking, pd.DataFrame(steps)

def top(ranking, k=5):
    return pd.DataFrame([{"rank": i + 1, "title": catalog[t]["track_name"][0], "artist": ", ".join(catalog[t]["artist_name"]),
                          "tempo": catalog[t]["tempo"][0] if catalog[t]["tempo"] else None,
                          "tags": ", ".join(catalog[t]["tag_list"][:5])} for i, t in enumerate(ranking[:k])])

A request for calm, slow piano music has a fact in it (slow) and a description (calm, piano). The fact goes to SQL, which keeps every track under 80 BPM; the description goes to dense retrieval over the tags, which reranks those tracks.

In [9]:
ranking, steps = run_tools([
    {"name": "sql", "arguments": {"sql_query": "SELECT track_id FROM tracks WHERE tempo < 80", "topk": 1000}},
    {"name": "text_to_item_similarity", "arguments": {"query": "calm piano", "corpus_type": "attributes", "topk": 20}},
])
display(steps)
top(ranking)

,tool,arguments,returned,error
0,sql,"{'sql_query': 'SELECT track_id FROM tracks WHERE tempo < 80', 'topk': 1000}",1000,
1,text_to_item_similarity,"{'query': 'calm piano', 'corpus_type': 'attributes', 'topk': 20}",20,


,rank,title,artist,tempo,tags
0,1,Week No. 10,Fabrizio Paterlini,77.90,"calm, classical, classical, piano"
1,2,Fragments Found,Fabrizio Paterlini,77.14,"instrumental, classical, piano"
2,3,Night,"Ludovico Einaudi, Amsterdam Sinfonietta",53.44,"classical, piano"
3,4,I Miss You,Ozymandias,76.28,"classical, piano"
4,5,Homemade Mountains,Christina Vantzou,59.75,"instrumental, piano, ambient, instrumental, melancholy"


Swap the order and the result changes. Dense retrieval now keeps only its top 20 over the whole catalog, and SQL can only remove tracks from those 20: it keeps 13. Worse, a filter has no notion of relevance, so it returns the survivors in table order and the ranking from the first step is lost. The order is part of the plan, which is why the LLM is asked for a retrieval stage that casts a wide net and a reranking stage that orders it.

In [10]:
ranking, steps = run_tools([
    {"name": "text_to_item_similarity", "arguments": {"query": "calm piano", "corpus_type": "attributes", "topk": 20}},
    {"name": "sql", "arguments": {"sql_query": "SELECT track_id FROM tracks WHERE tempo < 80", "topk": 20}},
])
display(steps)
top(ranking)

,tool,arguments,returned,error
0,text_to_item_similarity,"{'query': 'calm piano', 'corpus_type': 'attributes', 'topk': 20}",20,
1,sql,"{'sql_query': 'SELECT track_id FROM tracks WHERE tempo < 80', 'topk': 20}",13,


,rank,title,artist,tempo,tags
0,1,I Miss You,Ozymandias,76.28,"classical, piano"
1,2,The Earth Prelude,Ludovico Einaudi,52.31,"soundtracks, classical, piano, classical"
2,3,Fragments Found,Fabrizio Paterlini,77.14,"instrumental, classical, piano"
3,4,Nocturne in a Minor,Chad Lawson,64.66,"classical, piano, rock"
4,5,Night,"Ludovico Einaudi, Amsterdam Sinfonietta",53.44,"classical, piano"


## Step 5: Designing the agent

The agent turns a conversation into tool calls and tool results into a reply. TalkPlay-Tools does this with two LLM calls per turn. The first reads the conversation with the tool schemas and plans: Qwen3 thinks first, then writes its tool calls as JSON inside `<tool_call>` tags, which its chat template defines. The executor from Step 4 runs them. The second call reads the top track's description and writes the reply. Both calls share a system prompt assembled from three parts, taken from the repository: a **role**, the listener's **profile**, and the **goal** of the call. The conversation, with every recommended track, is the agent's **memory**.

:::{figure} ../assets/tools-agent.svg
:label: fig-tools-agent
:alt: The system prompt is assembled from a role, the listener profile (demographics, and for a returning listener five tracks from the last session with semantic IDs) and a goal. Call 1 adds the tool schemas and the memory, thinks, and writes tool calls. The executor runs them and describes the top track. Call 2 adds that description and writes the reply. Both turns are appended to the memory.

The agent. Two LLM calls per turn around the executor, with one system prompt assembled from role, profile and goal.
:::

In [11]:
ROLE = ("You are an expert music recommendation assistant. Your task is to understand user preferences and provide "
        "personalized music recommendations. Your thinking/reasoning process short, concise, but clear and complete.")
PERSONALIZATION = "Consider the following user profile and recommend personalized music tracks for the user."
GOAL_TOOL_CALLING = """You are tasked with executing an efficient music recommendation pipeline. Follow these steps precisely:
STAGE 1 - PLANNING:
- Analyze the user's query to determine the optimal tool combination.
- Provide short/brief reasoning for your tool choices
STAGE 2 - RETRIEVAL:
- Use ONE appropriate retrieval tool to gather candidate tracks
- Aim to collect at least 20 unique track_ids
STAGE 3 - RERANKING:
- Use ONE reranking tool to optimize the final selection
- Use the same topk value for both retrieval and reranking stages
REQUIREMENTS:
- Default setup: use total two tools for retrieval and reranking. (retrieval -> reranking)
- Cascading Filtering: When selecting retrieval and reranking tools, ensure they are complementary with NO overlap since the process is sequential and the results from the first tool affect the next tool.
- Top K: Use the same topk value for both retrieval and reranking stages to maintain consistent ranking depth
- Tool Selection: Choose the most effective tool for each stage based on the user's query type
- Final Recommendation: Ensure the final output contains at least 20 track recommendations
PERSONALIZATION: For reranking stage, prioritize user_to_item_similarity when user_type is "warm_start". For "cold_start" users, don't use user_to_item_similarity."""
GOAL_RESPONSE = """Based on the user query and the recommended track from tool calling results, provide a brief response that:
1. MUST base your response on the previously recommended track from the tool calling results
2. If the recommended track doesn't match the user's query, apologize and acknowledge the mismatch. It is problem of recommendation system.
3. If it's a good match, acknowledge that you've found music that matches their request with enthusiasm and confidence
4. Share key details including title, artist, and relevant musical information (genre, mood, style, or notable characteristics)
5. Briefly explain why this track is a good match for their specific request or preferences (or apologize if it's not a good match)
6. Invite further interaction by asking if they'd like to explore similar tracks, need recommendations for different moods, or have any other music preferences to discuss"""

In [12]:
class Agent:
    def __init__(self, user):
        self.user, self.memory = user, []

    def profile(self):
        u = self.user
        s = (f"{PERSONALIZATION}\n[DEMOGRAPHIC INFORMATION]\n user_id: {u['user_id']}, user_type: {u['user_type']}, "
             f"age_group: {u['age_group']}, gender: {u['gender']}")
        history = [t for t in u.get("last_track_ids", []) if t in catalog][:5]
        if history:
            s += (f"\n[PREVIOUS SESSION EXAMPLES]: Examples of tracks that the user_id: {u['user_id']} has listened to "
                  "in the last session:\n" + "\n".join(describe(t, semantic_ids=True) for t in history))
        return s

    def chat(self, message, plan_tokens=2048):
        self.memory.append({"role": "user", "content": message})
        start = time.time()
        system = [{"role": "system", "content": ROLE + "\n" + self.profile() + "\n" + GOAL_TOOL_CALLING}]
        plan = generate(system + self.memory, plan_tokens, tools=SCHEMAS, enable_thinking=True)
        thinking, _, calls_text = plan.rpartition("</think>")
        thinking = thinking.replace("<think>", "")
        calls = parse_tool_calls(calls_text or plan)
        ranking, steps = run_tools(calls)
        if not ranking:
            self.memory.pop()
            return {"message": message, "thinking": thinking.strip(), "steps": steps, "ranking": [], "reply": "", "seconds": time.time() - start}
        system = [{"role": "system", "content": ROLE + "\n" + self.profile() + "\n" + GOAL_RESPONSE}]
        grounding = [{"role": "assistant", "content": describe(ranking[0])}]
        reply = generate(system + self.memory + grounding, 400, enable_thinking=False).strip()
        self.memory.append({"role": "assistant", "content": f"recommended track: {describe(ranking[0])}\n{reply}"})
        return {"message": message, "thinking": thinking.strip(), "steps": steps, "ranking": ranking,
                "reply": reply, "seconds": time.time() - start}

def show(r):
    print("USER:", r["message"])
    print("\nTHINKING:", textwrap.shorten(r["thinking"], 700))
    display(r["steps"])
    if r["ranking"]:
        display(top(r["ranking"], 3))
        print(f"https://open.spotify.com/track/{r['ranking'][0]}")
        print("\nASSISTANT:", textwrap.fill(r["reply"], 100, replace_whitespace=False))
    else:
        print("\nNo tool returned a track, so there is nothing to recommend and no reply.")
    print(f"\n({r['seconds']:.0f} s)")

The profile is where personalization enters. For a new listener, a cold-start user, it holds demographics only. For a returning listener it also lists five tracks from the last session, each with its semantic IDs, so that the LLM can ask `semantic_id_matching` for tracks with similar codes, and the goal tells it to rerank with `user_to_item_similarity`, the collaborative-filtering tool.

In [13]:
newcomer = {"user_id": "NEW_USER", "user_type": "cold_start", "age_group": "20s", "gender": "male", "last_track_ids": []}
regular = next(u for u in users.values() if u["user_type"] == "warm_start" and len(u["last_track_ids"]) >= 5
               and u["user_id"] in USER_VECTORS)
print(Agent(newcomer).profile())
print()
lines = Agent(regular).profile().splitlines()
print("\n".join(lines[:5]))  # the header and the first of five tracks, in full
print(f"... and {len(lines) - 5} more tracks")

Consider the following user profile and recommend personalized music tracks for the user.
[DEMOGRAPHIC INFORMATION]
 user_id: NEW_USER, user_type: cold_start, age_group: 20s, gender: male

Consider the following user profile and recommend personalized music tracks for the user.
[DEMOGRAPHIC INFORMATION]
 user_id: 21362, user_type: warm_start, age_group: 30s, gender: male
[PREVIOUS SESSION EXAMPLES]: Examples of tracks that the user_id: 21362 has listened to in the last session:
track_id: 3lGMtkONrZdJ8kTCg6KIFf, title: hasta la raíz, artist: natalia lafourcade, album: hasta la raíz, tags: latin pop, latin, tempo: 101.84, key: Eb major, release_date: 2015-11-13, audio:semantic_id: [37, 44, 26, 5], image:semantic_id: [42, 8, 22, 62], metadata:semantic_id: [48, 4, 63, 42], lyrics:semantic_id: [28, 0, 26, 28], attributes:semantic_id: [13, 55, 40, 7], cf_item:semantic_id: [46, 17, 18, 52]
... and 4 more tracks


## Step 6: One request through the system

Now the LLM. We load Qwen3 and send the request the repository uses as its example, from the new listener: the plan, the tool calls with how many tracks each returned, the top three tracks, and the reply.

:::{figure} img/search-talkplay-tools-trace.png
:label: fig-demo-talkplay-trace
:width: 85%
:alt: One turn of the TalkPlay-Tools agent in three steps. Step 1, chain-of-thought: the model reasons about the request and the listening history and chooses BM25 for retrieval and user-to-item similarity for reranking. Step 2, the two tool calls as JSON. Step 3, the recommended track with its tags and a playable preview.

What one turn looks like in the full system: (01) the plan, (02) the tool calls, (03) the recommended track. The cell below prints the same three parts for our request. Figure adapted from {cite}`doh_talkplaytools_2025`.
:::

In [14]:
tokenizer = AutoTokenizer.from_pretrained(LM_ID)
lm = AutoModelForCausalLM.from_pretrained(LM_ID, dtype=LM_DTYPE).to(DEVICE).eval()
SAMPLING = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)  # Qwen3's recommended thinking settings

def generate(messages, max_new_tokens, **template_kwargs):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **template_kwargs)
    x = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        y = lm.generate(**x, max_new_tokens=max_new_tokens, **SAMPLING)
    return tokenizer.decode(y[0, x.input_ids.shape[1]:], skip_special_tokens=True)

def parse_tool_calls(text):
    calls = []
    for block in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, flags=re.S):
        try:
            call = json.loads(block)
        except json.JSONDecodeError:
            continue
        if "name" in call and "arguments" in call:
            calls.append(call)
    return calls

In [15]:
torch.manual_seed(0)
agent = Agent(newcomer)
show(agent.chat("I'm looking for calm and slow tempo piano music."))

USER: I'm looking for calm and slow tempo piano music.

THINKING: Okay, let's tackle this user query. The user is a 20-something male looking for calm and slow tempo piano music. Since he's a cold_start user, I can't use user_to_item_similarity in the reranking stage. First, for retrieval, I need to find tracks that match "calm" and "slow tempo" with piano. The BM25 tool is good for lexical matching. I'll use corpus_type "attributes" because the user mentioned tempo and mood, which are attributes. The query should be "calm slow tempo piano". Let me set topk to 20 to get enough candidates. Then, for reranking, I should use text_to_item_similarity with the same topk of 20. The query remains "calm slow tempo piano", and corpus_type "attributes" again. [...]


,tool,arguments,returned,error
0,bm25,"{'query': 'calm slow tempo piano', 'corpus_type': 'attributes', 'topk': 20}",20,
1,text_to_item_similarity,"{'query': 'calm slow tempo piano', 'corpus_type': 'attributes', 'topk': 20}",20,


,rank,title,artist,tempo,tags
0,1,Week No. 10,Fabrizio Paterlini,77.90,"calm, classical, classical, piano"
1,2,Les Feuilles Mortes (Autumn Leaves),Ernesto Cortazar,66.14,piano
2,3,Slow Jam,Euge Groove,75.54,"slow, easy listening, easy listening, piano, lounge"


https://open.spotify.com/track/30jCZxVZRYU5PlcR3z1A32

ASSISTANT: I've found some calming and slow tempo piano music that should match your request! The track **"Week
No. 10"** by **Fabrizio Paterlini** is a gentle and soothing piece with a tempo of 77.9 BPM, making
it perfect for a calm and relaxed atmosphere. It falls under the classical and piano genres,
featuring a soft and melodic style that's ideal for unwinding.

This track is a great match for your
request of calm and slow tempo piano music. If you'd like to explore similar tracks, need
recommendations for different moods, or have any other music preferences to discuss, feel free to
let me know!

(158 s)


The same request from the returning listener. The request is identical, and only the profile in the system prompt changes: it now holds five tracks from the last session, and the goal asks for `user_to_item_similarity` in the reranking stage.


In [16]:
torch.manual_seed(0)
returning = Agent(regular)
show(returning.chat("I'm looking for calm and slow tempo piano music."))

USER: I'm looking for calm and slow tempo piano music.

THINKING: Okay, let's tackle this user query. The user is asking for calm and slow tempo piano music. First, I need to figure out the best approach using the tools provided. The user's previous listens include tracks by Massive Attack and Madonna, which are more experimental and alternative, but the current request is for calm, slow piano. So, the user might be looking for a more relaxing vibe. For the retrieval stage, I should use a tool that can find tracks based on keywords. BM25 is good for lexical matching, so maybe using BM25 with the query "calm slow piano" on the attributes corpus. Attributes include genres, moods, etc., which would fit. Then, for reranking, since the user is a [...]


,tool,arguments,returned,error
0,bm25,"{'query': 'calm slow piano', 'corpus_type': 'attributes', 'topk': 20}",20,
1,user_to_item_similarity,"{'user_id': '21362', 'topk': 20}",20,


,rank,title,artist,tempo,tags
0,1,White Lake,Deaf Center,61.70,"experimental, alternative, love, summer, emotional"
1,2,Goodbye Pork Pie Hat,Charles Mingus,53.91,"slow, jazz, classical, piano, sad"
2,3,The Greatest Man That Ever Lived (Variations On A Shaker Hymn),Weezer,118.21,"slow, alternative rock, 00s, piano, awesome"


https://open.spotify.com/track/6lNUewdE3ZY4vUMxXpHtIC

ASSISTANT: I've found a perfect match for you! The track **"white lake"** by **deaf center** is a calm and slow
tempo piano piece that fits your request perfectly. It's an experimental, atmospheric, and
melancholic track with a gentle piano melody that's ideal for relaxing and unwinding. The tempo of
61.7 BPM and key of Bb minor create a soothing and introspective mood.

This track is a great choice
for its peaceful and reflective vibe. Would you like to explore similar tracks, or need
recommendations for different moods? Let me know!

(214 s)


Three things to notice in the two runs.

- **The plan is not the one we wrote by hand.** In Step 4 we filtered on `tempo < 80` with SQL. The agent treated "slow tempo" as words to match and sent them to BM25. The top tracks happen to be slow, but nothing in its plan guarantees it.
- **The profile changed the plan and the answer.** For the new listener the agent reranked with `text_to_item_similarity`. For the returning listener it followed the goal and reranked with `user_to_item_similarity`, so the same 20 candidates were ordered by taste and not by the request. The third track runs at 118 BPM.
- **The reply does not check the track.** It calls the top track a perfect match in both runs. The response prompt asks for enthusiasm, and whether the track is calm piano music is never verified.


## Conclusion

The agent never saw the catalog. It saw six docstrings, a profile, and the conversation, and turned the request into a two-step plan that the tools executed over precomputed indices. This is the division of labor argued for on the [search and recommendation page](./search-recommendation.md): the LLM reads the request and the listener, and the tools hold the catalog and the listening data.

Each layer can be changed on its own. In the environment, add the two tools left out here, text-to-audio and text-to-image search, which need the CLAP and SigLIP2 text encoders at query time; the repository has both. In the workflow, allow more than two stages, or let the LLM see the results of one call before it writes the next. In the agent, change a docstring or a goal and watch the plan change, or send a follow-up such as "more like that one, but with vocals" to the same `agent`, whose memory holds the `track_id` that `item_to_item_similarity` needs.

## Resources

- TalkPlay-Tools: [code](https://github.com/talkpl-ai/talkplay-tools), [paper](https://arxiv.org/abs/2510.01698), [tool environments](https://huggingface.co/datasets/talkpl-ai/TalkPlayTools-Env)
- TalkPlayData-2: [dataset](https://huggingface.co/datasets/talkpl-ai/TalkPlayData-2), [project page](https://talkpl.ai/talkplaydata2.html)
- [Qwen3 function calling](https://qwen.readthedocs.io/en/latest/framework/function_call.html) and [Qwen3-Embedding](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)
- [Tool use in 🤗 Transformers](https://huggingface.co/docs/transformers/chat_extras)